In [0]:
parquet_path = "abfss://datalake-elias@eliasadlsdev.dfs.core.windows.net/processed_Elias/AzureSQL/AzureSQL_Sales_Processed.parquet"

df = spark.read.parquet(parquet_path)

display(df)

In [0]:
from pyspark.sql.functions import col, when

df_transformed = (
    df
    .withColumn(
        "SalesStatus",
        when(col("SalesAmount") >= 1000, "High")
        .otherwise("Low")
    )
)

display(df_transformed)

In [0]:
from pyspark.sql.functions import col, abs as spark_abs

SEP = "=" * 60

# ── 1. Null checks across all columns ────────────────────────────
print(SEP)
print("1. NULL VALUE COUNTS PER COLUMN")
print(SEP)
null_counts = [(c, df.filter(col(c).isNull()).count()) for c in df.columns]
for col_name, cnt in null_counts:
    status = "FAIL" if cnt > 0 else "PASS"
    print(f"  [{status}] {col_name}: {cnt} null(s)")

# ── 2. Duplicate SaleID ──────────────────────────────────────────
print(f"\n{SEP}")
print("2. DUPLICATE SaleID")
print(SEP)
df_dup_saleid = (
    df.groupBy("SaleID")
      .count()
      .filter(col("count") > 1)
)
dup_count = df_dup_saleid.count()
if dup_count > 0:
    print(f"  [FAIL] {dup_count} duplicate SaleID(s) found:")
    display(df_dup_saleid)
else:
    print("  [PASS] No duplicate SaleID values")

# ── 3. Invalid Quantity (<= 0) ───────────────────────────────────
print(f"\n{SEP}")
print("3. INVALID QUANTITY (<= 0)")
print(SEP)
df_bad_qty = df.filter(col("Quantity") <= 0)
bad_qty_count = df_bad_qty.count()
if bad_qty_count > 0:
    print(f"  [FAIL] {bad_qty_count} record(s) with Quantity <= 0:")
    display(df_bad_qty)
else:
    print("  [PASS] All Quantity values are > 0")

# ── 4. Invalid UnitPrice (<= 0) ──────────────────────────────────
print(f"\n{SEP}")
print("4. INVALID UnitPrice (<= 0)")
print(SEP)
df_bad_price = df.filter(col("UnitPrice") <= 0)
bad_price_count = df_bad_price.count()
if bad_price_count > 0:
    print(f"  [FAIL] {bad_price_count} record(s) with UnitPrice <= 0:")
    display(df_bad_price)
else:
    print("  [PASS] All UnitPrice values are > 0")

# ── 5. SalesAmount == Quantity * UnitPrice ───────────────────────
print(f"\n{SEP}")
print("5. SalesAmount == Quantity * UnitPrice")
print(SEP)
df_bad_amount = df.filter(
    spark_abs(col("SalesAmount") - (col("Quantity") * col("UnitPrice"))) > 0.01
)
bad_amount_count = df_bad_amount.count()
if bad_amount_count > 0:
    print(f"  [FAIL] {bad_amount_count} record(s) where SalesAmount != Quantity * UnitPrice:")
    display(df_bad_amount)
else:
    print("  [PASS] SalesAmount matches Quantity * UnitPrice for all records")

# ── Summary ──────────────────────────────────────────────────────
print(f"\n{SEP}")
print("DATA QUALITY SUMMARY")
print(SEP)
total_rows = df.count()
total_nulls = sum(cnt for _, cnt in null_counts)
print(f"  Total rows        : {total_rows}")
print(f"  Null values       : {total_nulls}")
print(f"  Duplicate SaleIDs : {dup_count}")
print(f"  Invalid Quantity  : {bad_qty_count}")
print(f"  Invalid UnitPrice : {bad_price_count}")
print(f"  Amount mismatch   : {bad_amount_count}")

In [0]:
output_path = "abfss://datalake-elias@eliasadlsdev.dfs.core.windows.net/curated_Elias/Sales/"

df_transformed.write \
    .mode("overwrite") \
    .parquet(output_path)

print("Successfully written to curated_Elias/Sales/")